# Tutorial on donut radius fitting

* **lsst_distrib**  `w_2025_12`
* **ts_wep**  `v14.2.0`

In this notebook we illustrate various methods for donut radius fitting.

## Imports

In [ ]:
from lsst.daf.butler import Butler
from lsst.ts.wep.task.generateDonutDirectDetectTask import (
GenerateDonutDirectDetectTask,GenerateDonutDirectDetectTaskConfig)
from lsst.ts.wep.task.cutOutDonutsScienceSensorTask import (CutOutDonutsScienceSensorTask, CutOutDonutsScienceSensorTaskConfig)
from lsst.daf.butler import Butler
from lsst.ts.wep.task.cutOutDonutsCwfsTask import (CutOutDonutsCwfsTask, CutOutDonutsCwfsTaskConfig)
from lsst.ts.wep.task.fitDonutRadiusTask import (FitDonutRadiusTask, FitDonutRadiusTaskConfig)
from lsst.obs.lsst import LsstCam
import matplotlib.pyplot as plt 
import numpy as np 
from lsst.obs.lsst import LsstCam
from scipy.optimize import curve_fit
from scipy.signal import find_peaks
from scipy.ndimage import gaussian_filter

## Plotting and fitting functions

In [ ]:
def fit_radius(
    image,
    multiplier=0.8,
    filter_sigma=3,
    min_peak_width=5,
    min_height=0.3,
    left_default_perc=0.1,
    right_default_perc=0.8,
    cross_section=True,
    marginalization=False
):
    """
    Estimate the radius of a donut-shaped object by analyzing a cross-section
    of its image using peak detection on a smoothed intensity profile.

    The function normalizes and smooths the central horizontal cross-section
    of the image, then uses a peak-finding algorithm to detect the inner and
    outer edges of the donut. If peak detection fails or produces invalid
    edges, default values based on image size are used.

    Parameters
    ----------
    image : numpy.ndarray
        A 2D array representing the donut stamp image.
    multiplier : float, optional
        Multiplier used to convert the width of detected peaks to edge offsets.
        Default is 0.8.
    filter_sigma : float, optional
        Standard deviation (in pixels) of the Gaussian kernel used to smooth
        the cross-section before peak detection. Default is 3.
    min_peak_width : float, optional
        Minimum allowed width (in pixels) for peaks in the cross-section.
        Default is 5.
    min_height : float, optional
        Minimum height of peaks in the normalized cross-section
        (value between 0 and 1). Default is 0.3.
    left_default_perc : float, optional
        Fallback value for the left edge as a fraction of image width
        if peak detection fails. Default is 0.1.
    right_default_perc : float, optional
        Fallback value for the right edge as a fraction of image width
        if peak detection fails. Default is 0.8.

    Returns
    -------
    filtered_x_profile : numpy.ndarray
        The smoothed, normalized horizontal cross-section used for peak detection.
    peak_locations : numpy.ndarray
        The x-positions of detected peaks in the filtered cross-section.
    peak_information : dict
        Properties of detected peaks returned by `scipy.signal.find_peaks`,
        including widths and heights.
    left_edge : float
        The estimated position of the left edge of the donut.
    right_edge : float
        The estimated position of the right edge of the donut.
    radius : float
        The estimated radius of the donut, defined as half the distance between
        left and right edges.
    fail_flag : int
        Flag indicating if fallback/default values were used.
        1 indicates a failure in peak detection or invalid edge values;
        0 indicates success.
    """
    half_width = int(len(image) / 2)
    if cross_section:
        y_cross = image[halfWidth,:]
        y_cross_norm = np.array(y_cross) / max(y_cross)
    if marginalization:
        y_cross = image.sum(axis=0)  # Collapse along y-axis → marginalize onto x-axis
        y_cross_norm = y_cross / np.max(y_cross)  # Normalize to [0, 1]

    fail_flag = 0

    # Convolve with Gaussian filter to smooth out the cross-section
    filtered_x_profile = gaussian_filter(y_cross_norm, sigma=filter_sigma)

    # Defaults used in case of fit failure
    left_default_edge = left_default_perc * len(image)
    right_default_edge = right_default_perc * len(image)

    # Detect peaks
    peak_locations, peak_information = find_peaks(
        filtered_x_profile,
        height=min_height,
        width=min_peak_width,
    )

    if len(peak_locations) > 0:
        # Choose left and right peaks
        index_of_right = np.argmax(peak_locations)
        index_of_left = np.argmin(peak_locations)

        left_width = peak_information["widths"][index_of_left]
        right_width = peak_information["widths"][index_of_right]
        left_peak = peak_locations[index_of_left]
        right_peak = peak_locations[index_of_right]

        left_edge = left_peak - left_width * multiplier
        right_edge = right_peak + right_width * multiplier
    else:
        left_edge = left_default_edge
        right_edge = right_default_edge
        fail_flag = 1
        print(f"Setting left edge to {left_edge} and right edge to {right_edge}")

    # Catch successful fit with bad values
    if left_edge < 0:
        print(f"Warning: left_edge < 0 ({left_edge}), using default.")
        left_edge = left_default_edge
        fail_flag = 1

    if right_edge > len(image):
        print(f"Warning: right_edge > image size ({right_edge}), using default.")
        right_edge = right_default_edge
        fail_flag = 1

    # Donut radius is half of the distance between the two edges
    radius = (right_edge - left_edge) / 2.0

    return (
        filtered_x_profile,
        peak_locations,
        peak_information,
        left_edge,
        right_edge,
        radius,
        fail_flag,
    )




# Define a function for two Gaussians
def double_gaussian(x, A1, mu1, sigma1, A2, mu2, sigma2):
    return (A1 * np.exp(-((x - mu1) ** 2) / (2 * sigma1 ** 2)) +
            A2 * np.exp(-((x - mu2) ** 2) / (2 * sigma2 ** 2)))

def fit_radius_double_gaussian(image, initial_guess= [0.8, 50, 20, 0.5, 150, 25], sigma_cutoff = 1.75,
                               left_default_perc=0.1, right_default_perc=0.8,
                              cross_section=True, marginalization=False):
    """
    Fits a cross-section of a 2D image with a sum of two Gaussian profiles 
    and estimates the effective radius between their edges.

    Parameters
    ----------
    image : 2D array-like
        A square 2D image array from which the central horizontal cross-section is extracted.
    initial_guess : list of float, optional
        Initial parameter estimates for the double Gaussian fit. 
        Format: [A1, mu1, sigma1, A2, mu2, sigma2], where A is amplitude, 
        mu is mean position, and sigma is the standard deviation.
    sigma_cutoff : float, optional
        Multiplier of the standard deviation used to determine the "edge" 
        of each Gaussian. Default is 1.75 (captures ~91% of the Gaussian area).
    left_default_perc: float
        The position of the left edge of the donut in case
        fitting failed: a fraction of image length
        (between 0 and 1). Default is 0.1.
    right_default_perc: float
        The position of the right edge of the donut in case
        fitting failed: a fraction of image length
        (between 0 and 1). Default is 0.8.

    Returns
    -------
    x_pos : ndarray
        The x-axis positions corresponding to the cross-section.
    y_cross_norm : ndarray
        The normalized intensity values from the central horizontal cross-section.
    y_fit : ndarray
        The fitted values from the double Gaussian model.
    radius : float
        Half the distance between the left and right Gaussian edges,
        interpreted as an effective "radius".
    x_left_edge : float
        The estimated left boundary based on the smaller-radius Gaussian.
    x_right_edge : float
        The estimated right boundary based on the larger-radius Gaussian.

    Notes
    -----
    The function models the image cross-section as a sum of two Gaussians
    and estimates the spatial spread between them based on a cutoff of 
    `sigma_cutoff * sigma`. This can be useful in image analysis involving
    symmetric structures or dual-lobed features.

    If the fit fails, fallback values (`left_default_edge`, `right_default_edge`)
    must be defined in the global scope. A warning will be printed.

    Raises
    ------
    RuntimeError
        If the curve fitting fails, although the exception is caught internally
        and fallback edge values are used.
    """
    halfWidth = int ( len(image) / 2 )
    x_pos = np.arange(0,len(image))

    if cross_section:
        y_cross = image[halfWidth,:]
        y_cross_norm = np.array(y_cross) / max(y_cross)
    if marginalization:
        y_cross = image.sum(axis=0)  # Collapse along y-axis → marginalize onto x-axis
        y_cross_norm = y_cross / np.max(y_cross)  # Normalize to [0, 1]


    filtered_x_profile = gaussian_filter(y_cross_norm, sigma=3)
    
    # set the defaults used in case of fit failure,
    # so that the returned radius will be reasonable
    left_default_edge = left_default_perc * len(image)
    right_default_edge = right_default_perc * len(image)
    y_fit = np.zeros(len(y_cross))
    try:
        # Fit the data
        x = x_pos
        y_noisy = filtered_x_profile
        popt, pcov = curve_fit(double_gaussian, x, y_noisy, p0=initial_guess)
        
        # Extract fitted parameters
        A1_fit, mu1_fit, sigma1_fit, A2_fit, mu2_fit, sigma2_fit = popt
        
        # Generate fitted curve
        y_fit = double_gaussian(x, *popt)
    
        # figure out which is one with  larger position along the cross-section
        mu_values =  [mu1_fit, mu2_fit]
        sigma_values = [sigma1_fit, sigma2_fit]
    
        # find out the position of the Gaussian at larger radius     
        index_of_larger = np.argmax(mu_values) 
    
        mu_larger = mu_values[index_of_larger]
        sigma2 = sigma_values[index_of_larger] # take the sigma corresponding to that mu 
    
        # right edge of Gaussian at a larger radius
        x_right_edge = mu_larger+sigma_cutoff*sigma2
    
        # find out the position of the Gaussian at smaller radius
        index_of_smaller = np.argmin(mu_values)
    
        mu_smaller = mu_values[index_of_smaller]
        sigma1 = sigma_values[index_of_smaller]
    
        # left edge of Gaussian at a smaller radius
        x_left_edge  = mu_smaller - sigma_cutoff*sigma1
        
    except RuntimeError as e:
        x_left_edge = left_default_edge
        x_right_edge = right_default_edge
        fail_flag = 1
        print(
                f"Setting left edge to {x_left_edge} and right edge to {x_right_edge}",
            e
            )
  
    radius = (x_right_edge - x_left_edge ) / 2.

    return x_pos, y_cross_norm, y_fit, radius, x_left_edge, x_right_edge, filtered_x_profile

def find_peak_edge_dropoffs(profile, percentile=5.0):
    """
    Find the positions where a 1D profile falls to a specified percentile 
    of the two highest peaks, on the side closest to the respective edge.

    Parameters:
    -----------
    profile : np.ndarray
        1D array representing the profile (e.g., intensity across a line).
    percentile : float, optional
        Percentile (0-100) of the peak height to define the threshold. 
        Default is 5.0 (i.e., 5%).

    Returns:
    --------
    left_edge : float
        Pixel x-position where the provided profile falls below the 
        given percentile of the highest peak closest to the left edge.

    right_edge : float
        Pixel x-position where the provided profile falls below the
        given percentile of the highest peak closest to the right edge.

    left_peak : float
        The local peak value used near the left edge.

    right_peak : float
        The local peak value used near the right edge.

    left_peak_height : float
        The height of the left peak.

    right_peak : float
        The height of the right peak.
    """
    x = np.arange(len(profile))
    min_peak_width=5
    min_height=0.3
 
    peak_locations, peak_information = find_peaks(filtered_x_profile,  height=min_height, width=min_peak_width,
                                             prominence=0.2)
    if len(peak_locations) < 2:
       raise ValueError("Need at least two peaks in the profile.")

    index_of_right = np.argmax(peak_locations)
    index_of_left = np.argmin(peak_locations)

    left_width = peak_information["widths"][index_of_left]
    right_width = peak_information["widths"][index_of_right]
    
     
    left_peak = peak_locations[index_of_left]
    right_peak = peak_locations[index_of_right]

    # Set threshold as percentage of peak height 
    left_peak_height = profile[left_peak]
    right_peak_height = profile[right_peak]

    left_threshold = (percentile / 100.0) * left_peak_height
    right_threshold = (percentile / 100.0) * right_peak_height


    if min(profile[:left_peak]) >= left_threshold:
        left_edge = x[0]
    else:
        # left edge is the last index where the left part of the 
        # profile is smaller 
        left_edge = np.where(profile[:left_peak] < left_threshold)[0][-1]
    
    if min(profile[right_peak:]) >= right_threshold:
        right_edge = x[-1]
    else:
        # right edge is the first index where the right part 
        # of the profile is smaller 
        right_edge_relative = np.where(profile[right_peak:] < right_threshold)[0][0]
        right_edge = right_edge_relative + right_peak


    return left_edge, right_edge, left_peak, right_peak, left_peak_height, right_peak_height
    
def plot_donut_fits(
    stamps,
    nrows=4,
    ncols=4,
    w=3,
    widthMultiplier=0.8,
    filterSigma=3,
    minPeakWidth=5,
    minPeakHeight=0.3,
):
    """
    Plot donut image stamps along with two methods for estimating donut radii.

    This function displays a grid of plots where each column corresponds to a 
    single donut stamp. Each column shows:
        - The original image (row 0)
        - A 1D cross-section with a double-Gaussian fit (row 1)
        - A filtered 1D cross-section with peaks detected using scipy (row 2)

    Parameters
    ----------
    stamps : list
        A list of stamp objects. Each object must have a `stamp_im.image.array` attribute.
        Optionally, `stamps.metadata` can include 'VISIT', 'DET_NAME', and 'DFC_TYPE' 
        for figure titling.
    
    nrows : int, optional
        Number of rows in the subplot grid. Defaults to 3 (image, double-Gaussian fit, peak detection).

    ncols : int, optional
        Number of columns in the subplot grid. Each stamp occupies one column. Default is 4.
    
    w : float, optional
        Width scaling factor for figure size. The overall figure size is (ncols * w, nrows * w).
        Default is 3.

    widthMultiplier : float, optional
        Parameter passed to `fit_radius()` for controlling peak width multiplier. Default is 0.8.
    
    filterSigma : float, optional
        Gaussian filter sigma used in `fit_radius()` for smoothing. Default is 3.

    minPeakWidth : float, optional
        Minimum peak width required in `fit_radius()`. Default is 5.

    minPeakHeight : float, optional
        Minimum peak height threshold in `fit_radius()`. Default is 0.3.

    Returns
    -------
    fig : matplotlib.figure.Figure
        The matplotlib Figure object containing the plot.

    ax : numpy.ndarray of Axes
        2D array of matplotlib Axes objects (shape: [nrows, ncols]).

    Notes
    -----
    This function relies on two external functions:
        - `fit_radius_double_gaussian(image)`
        - `fit_radius(image, multiplier, filter_sigma, min_peak_width, min_height)`
    These must be defined in the user's environment for the function to work.

    """
    nstamps = len(stamps)
    if ncols > nstamps:
        ncols = nstamps
    fig, ax = plt.subplots(nrows, ncols, figsize=(ncols * w, nrows * w))
    ax = ax.reshape(nrows, ncols)
    radii_1 = []
    radii_2 = []
    radii_3 = []
    
    for i in range(ncols):
        col = i
        stamp = stamps[i]
        image = stamp.stamp_im.image.array

        # --- First row: original image ---
        row = 0
        ax[row, col].imshow(image, origin='lower')
        ax[row, col].set_xticks([])
        ax[row, col].set_yticks([])

        # --- Second row: Double Gaussian fit ---
        row = 1
        x_pos, y_cross_norm, y_fit, radius_dg, x_left_edge_dg, \
               x_right_edge_dg, filtered_x_profile = fit_radius_double_gaussian(image, cross_section=False, 
                                                                                marginalization=True)
        radii_1.append(radius_dg)
        ax[row, col].plot(x_pos, y_cross_norm, alpha=0.5, label='Donut profile')
        ax[row, col].plot(x_pos, filtered_x_profile, label='Filtered donut profile')
        ax[row, col].plot(x_pos, y_fit, '--', color='red', label='Fit')
        ax[row, col].vlines([x_left_edge_dg, x_right_edge_dg], ymin=0, ymax=1, color='red', lw=3)
        ax[row, col].set_title(f'r={radius_dg:.2f} px')

        # --- Third row: Scipy peak detection ---
        row = 2
        filtered_x_profile, peak_locations, peak_information, x_left_edge, x_right_edge, radius, flag = fit_radius(
            image=image,
            multiplier=widthMultiplier,
            filter_sigma=filterSigma,
            min_peak_width=minPeakWidth,
            min_height=minPeakHeight,
            cross_section=False, marginalization=True
        )
        radii_2.append(radius)
        ax[row, col].plot(filtered_x_profile, label='Filtered donut profile')
        ax[row, col].scatter(peak_locations, peak_information['peak_heights'], color='red', label='Peak locations')
        ax[row, col].vlines([x_left_edge, x_right_edge], ymin=0, ymax=1, color='red')
        ax[row, col].set_title(f'r={radius:.2f} px')

        # ---- Fourth row: more robust marginalized profile detection
        row=3
        dropoff_left, dropoff_right, left_peak, right_peak, \
           left_peak_height, right_peak_height = find_peak_edge_dropoffs(filtered_x_profile, percentile=5.0)
        radius = (dropoff_right - dropoff_left ) / 2.
        radii_3.append(radius)
        ax[row, col].plot(filtered_x_profile, label='Filtered donut profile')
        ax[row, col].vlines([dropoff_left, dropoff_right], ymin=0, ymax=1, color='red')
        ax[row, col].set_title(f'r={radius:.2f} px')
        
    # Hide x-ticks for rows 0 and 1
    for row in range(nrows-1):
        for col in range(ncols):
            ax[row, col].set_xticks([])

    # Add legends and labels
    row = 1
    col = ncols - 1
    ax[row, col].legend(bbox_to_anchor=[1.0, 0.65], title='Double Gaussian fit')
    ax[row, 0].set_ylabel('Normalized counts')

    row = 2
    ax[row, col].legend(bbox_to_anchor=[1.0, 0.65], title='Scipy Peak Locator')
    ax[row, 0].set_ylabel('Normalized counts')

    row = 3
    ax[row, col].legend(bbox_to_anchor=[1.0, 0.65], title='Robust edge finder')
    ax[row, 0].set_ylabel('Normalized counts')
    for col in range(ncols):
      ax[row, col].set_xlabel('x position [px]')

    fig.subplots_adjust(hspace=0.2, wspace=0.2)

    if hasattr(stamps, "metadata"):
        meta = stamps.metadata
        visit = meta.get('VISIT', 'UNKNOWN')
        det_name = meta.get('DET_NAME', 'UNKNOWN')
        dfc_type = meta.get('DFC_TYPE', 'UNKNOWN')
        fig.suptitle(f"{visit} {det_name}  {dfc_type} donuts")

    return fig, ax, radii_1, radii_2, radii_3

## Detect and cutout donuts for CWFS case

In [ ]:
butler = Butler('LSSTCam', collections=['LSSTCam/runs/quickLook'])
day_obs = 20250513
seq_num = 154
#det_ids = 191  # lower#  is extra-focal for Cwfs 
dataRefs = butler.query_datasets('post_isr_image' ,collections=['LSSTCam/runs/quickLook'],
                        where=f"instrument='LSSTCam' and exposure.day_obs={day_obs} and exposure.seq_num = {seq_num}\
                        and detector.purpose = 'WAVEFRONT' and detector.id in (191,192)"
                                )

exposure_extra = butler.get('post_isr_image', dataId=dataRefs[0].dataId)
exposure_intra = butler.get('post_isr_image', dataId=dataRefs[1].dataId)


camera = LsstCam.getCamera()

config = GenerateDonutDirectDetectTaskConfig()
config.instConfigFile = 'policy:instruments/LsstCam.yaml'
#config.donutSelector.useCustomMagLimit = True
task= GenerateDonutDirectDetectTask(config=config)
taskOutIntra =  task.run(exposure_intra,camera)

config = GenerateDonutDirectDetectTaskConfig()
config.instConfigFile = 'policy:instruments/LsstCam.yaml'
#config.doDonutSelection = False
#config.donutSelector.useCustomMagLimit = True
task= GenerateDonutDirectDetectTask(config=config)
taskOutExtra =  task.run(exposure_extra,camera)


config = CutOutDonutsCwfsTaskConfig() 
config.donutStampSize= 160
config.instConfigFile = 'policy:instruments/LsstCam.yaml'
task = CutOutDonutsCwfsTask(config=config)

taskCutOutIntra = task.run(exposure_intra, taskOutIntra.donutCatalog, camera) 
taskCutOutExtra = task.run(exposure_extra, taskOutExtra.donutCatalog, camera)



## Fit for radius

Fit with the fitDonutRadiusTask (using scipy peak locator), as well as  an alternative way (using Gaussian double peak fitting). Plot the cross-sections or marginalized donut images  (summing over the y-axis to get a 1D profile along x) used. This 1D data is further smoothed with a Gaussian kernel to reduce noise. 

The first approach uses scipy peak finder to find peak locations, and returns the peak position +/- fraction of peak width.  
The second approach fits a double Gaussian to the same data $ G(x) = A e^{-\frac{(x - \mu)^2}{2\sigma^2}} $, and returns peak position (mean of each Gaussian) +/-  fraction of its standard deviation.

The third approach takes the 1D profile, finds peaks with scipy peak finder, and find positions at which the value of the profile falls to `p%` (default `5%`) of peak height.

Illustrate the third approach for all donuts in extra and intra cutouts: 

In [ ]:
for stamps in [taskCutOutIntra.donutStampsOut , taskCutOutExtra.donutStampsOut]:
    fig,ax = plt.subplots(1,1,figsize=(7,5))
    i = 0
    
    for stamp in  stamps : 
        image = stamp.stamp_im.image.array
        halfWidth = int ( len(image) / 2 )
        x_pos = np.arange(0,len(image))
        
        # marginalize
        y_cross = image.sum(axis=0)  # Collapse along y-axis → marginalize onto x-axis
        y_cross_norm = y_cross / np.max(y_cross)  # Normalize to [0, 1]
        
        
        filtered_x_profile = gaussian_filter(y_cross_norm, sigma=3)
        
        ax.plot(filtered_x_profile, label=f'Donut {i} profile')
        left_edge, right_edge, left_peak, right_peak,  left_peak_height, \
        right_peak_height = find_peak_edge_dropoffs(filtered_x_profile, percentile=10.0)
        ax.vlines(left_edge,  ymin=0, ymax=1, color='red')
        ax.vlines(right_edge,  ymin=0, ymax=1, color='blue')
        ax.vlines([left_peak, right_peak], ymin=0, ymax=1,  color='k')
        i += 1
    ax.set_xlabel('position [px]')
    ax.set_ylabel('Normalized counts')
    ax.legend()
    ax.set_title(stamps.metadata['DFC_TYPE'])

We see that it arrives at very consistent values for the edge position.

That's for the intra-focal:

In [ ]:
_,_ , intra_radii_1, intra_radii_2, intra_radii_3 = plot_donut_fits(taskCutOutIntra.donutStampsOut, ncols=4, w=3)

In [ ]:
_,_,extra_radii_1, extra_radii_2, extra_radii_3  = plot_donut_fits(taskCutOutExtra.donutStampsOut, ncols=4, w=3)